# AI Replace Inpaint Smoke Runner

Standalone Pokecut-style AI Replace flow under `inpaint/`.

The Kaggle dataset presets live in `inpaint.smoke_runner`; this notebook only sets runtime knobs and calls the single all-preset entrypoint.


## 1. Install Dependencies


In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" safetensors ultralytics huggingface_hub opencv-python pillow numpy pandas matplotlib


## 2. Clone Or Update Repo


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git fetch origin main
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR if (REPO_DIR / "inpaint").exists() else Path.cwd()
print("PROJECT_DIR:", PROJECT_DIR)


## 3. Imports


In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/inpaint").exists() else Path.cwd()
sys.path = [str(PROJECT_DIR)] + [path for path in sys.path if path != str(PROJECT_DIR)]
%cd {PROJECT_DIR}

from inpaint.config import DEFAULT_CONFIG, AIReplaceConfig
from inpaint.smoke_runner import KAGGLE_DATASET_PRESETS, run_all_kaggle_presets

print("Loaded inpaint flow:", DEFAULT_CONFIG.AI_REPLACE_FLOW)
print("Model:", DEFAULT_CONFIG.MODEL_ID)


## 4. Runtime Check


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 2), "GB")


## 5. Hugging Face Login


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("No Kaggle HF_TOKEN secret found:", type(exc).__name__)

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")
else:
    print("No HF token provided; public models only")


## 6. Dataset Smoke Config


In [ ]:
SMOKE_IMAGES = 3
SEED = 42
DRY_RUN = False
USE_YOLO = True
OUTPUT_BASE = Path('/kaggle/working/ai_replace_smoke')

print("Kaggle presets loaded from inpaint.smoke_runner:")
for name, cfg in KAGGLE_DATASET_PRESETS.items():
    input_dir = Path(cfg["input_dir"])
    status = "found" if input_dir.exists() else "missing"
    print(f"- {name}: {status} | {input_dir}")


## 7. Smoke Helpers


In [ ]:
def show_combined_metrics(output_base=OUTPUT_BASE):
    summary_path = output_base / 'metrics' / 'metrics_summary.json'
    summary = json.load(open(summary_path, encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


def manifest_head(output_base=OUTPUT_BASE, n=20):
    import pandas as pd
    manifest_path = output_base / 'manifest.csv'
    if not manifest_path.exists():
        print('Manifest not found:', manifest_path)
        return None
    return pd.read_csv(manifest_path).head(n)


## 8. Optional Dry-Run Wiring Test


In [ ]:
dry_rows, dry_summary = run_all_kaggle_presets(
    output_dir=Path('/kaggle/working/ai_replace_dry_run'),
    num_images=1,
    seed=SEED,
    load_model=False,
    use_yolo=False,
    fail_on_missing=False,
)
dry_summary


## 9. Run All Kaggle Presets


In [ ]:
combined_rows, combined_summary = run_all_kaggle_presets(
    output_dir=OUTPUT_BASE,
    num_images=SMOKE_IMAGES,
    seed=SEED,
    load_model=not DRY_RUN,
    use_yolo=USE_YOLO,
    fail_on_missing=True,
)
combined_summary


## 10. Combined Metrics


In [ ]:
combined_metrics = show_combined_metrics()
manifest_head()


## 11. Preview Gallery


In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

PREVIEW_DATASET = "citypersons_bg_yolo"
preview_dir = OUTPUT_BASE / PREVIEW_DATASET / 'previews'
harmonized = sorted(preview_dir.glob('*_harmonized.png'))[:6]
if not harmonized:
    print('No previews found:', preview_dir)
else:
    cols = 3
    rows_n = math.ceil(len(harmonized) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(harmonized, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 12. Outside-Mask Diff Preview


In [ ]:
diffs = sorted((OUTPUT_BASE / PREVIEW_DATASET / 'previews').glob('*_diff_outside_mask.png'))[:6]
if not diffs:
    print('No diff previews found')
else:
    cols = 3
    rows_n = math.ceil(len(diffs) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(diffs, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 13. Export Outputs


In [ ]:
zip_base = Path('/kaggle/working') / OUTPUT_BASE.name
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_BASE)
print('Saved export:', zip_path)
